# ZAS Python Working Group — Session 2
## There Is Usually a Better Way

A very common pattern in scientific scripting: you solve a problem correctly with a loop,
then later discover Python (or NumPy) already has a built-in that does the same thing
in one line — faster, more readable, and less error-prone.

Each example below shows a **correct but verbose solution** followed by a **more elegant one**.
Both produce identical results — the point is style, readability, and efficiency.

---

In [ ]:
# Setup — run this first
import csv
import numpy as np

with open('data/reaction_times.csv', newline='') as f:
    trials = list(csv.DictReader(f))

# Pre-parse the columns we'll use repeatedly
rts        = [float(t['rt'])                   for t in trials]
conditions = [t['condition']                   for t in trials]
correct    = [t['correct'] == 'True'           for t in trials]
pids       = [t['participant_id']              for t in trials]

rts_np = np.array(rts)

print(f'Loaded {len(trials)} trials')

---
## Example 1 — Converting units (RT → milliseconds)

In [ ]:
# ❌  Verbose: manual for loop

rts_ms_loop = []
for rt in rts:
    rts_ms_loop.append(rt * 1000)

print(rts_ms_loop[:5])

In [ ]:
# ✅  Better: list comprehension

rts_ms_comp = [rt * 1000 for rt in rts]

print(rts_ms_comp[:5])

In [ ]:
# ✅✅  Best: NumPy vectorised arithmetic — no loop at all
#
# NumPy applies the operation to every element simultaneously.
# On large arrays this is orders of magnitude faster.

rts_ms_np = rts_np * 1000

print(rts_ms_np[:5])

# All three results are identical:
assert rts_ms_loop == rts_ms_comp
assert np.allclose(rts_ms_comp, rts_ms_np)
print('All three approaches agree ✓')

> **Why does it matter?**  
> On 400 trials the difference is invisible. On 400,000 EEG samples or a million fixations,
> the NumPy version runs in microseconds; the loop takes seconds.

---
## Example 2 — Mean RT per condition

In [ ]:
# ❌  Verbose: two separate loops, lots of bookkeeping

cong_rts   = []
incong_rts = []

for t in trials:
    if t['condition'] == 'congruent':
        cong_rts.append(float(t['rt']))
    else:
        incong_rts.append(float(t['rt']))

mean_cong   = sum(cong_rts)   / len(cong_rts)
mean_incong = sum(incong_rts) / len(incong_rts)

print(f'congruent:   {mean_cong:.3f} s')
print(f'incongruent: {mean_incong:.3f} s')

In [ ]:
# ✅  Better: dict comprehension — scales to any number of conditions
#
# Works even if you have 5 conditions and don't know their names in advance.

cond_names = set(conditions)

mean_per_condition = {
    cond: np.mean([rt for rt, c in zip(rts, conditions) if c == cond])
    for cond in cond_names
}

for cond, mean in sorted(mean_per_condition.items()):
    print(f'{cond:15s}: {mean:.3f} s')

> **Why does it matter?**  
> The verbose version requires you to add a new variable and a new `if` branch for every condition.
> The dict comprehension adapts automatically — same code works for 2 or 20 conditions.

---
## Example 3 — Counting correct trials

In [ ]:
# ❌  Verbose: loop with a counter variable

n_correct = 0
for c in correct:
    if c:
        n_correct += 1

print(f'Correct trials: {n_correct} / {len(trials)}')

In [ ]:
# ✅  Better: sum() over booleans
#
# In Python, True == 1 and False == 0, so sum() counts the Trues directly.

n_correct = sum(correct)

print(f'Correct trials: {n_correct} / {len(trials)}')

In [ ]:
# ✅✅  Also great: np.sum() — and gives you accuracy in the same line

correct_np = np.array(correct)

n_correct  = np.sum(correct_np)
accuracy   = np.mean(correct_np)   # mean of 0s and 1s = proportion correct

print(f'Correct trials: {n_correct} / {len(trials)}')
print(f'Accuracy:       {accuracy:.1%}')

> **Why does it matter?**  
> `sum(booleans)` is a Python idiom worth knowing — it shows up constantly in data work.
> `np.mean()` on a boolean array gives you a proportion for free, no division needed.

---
## Example 4 — Finding the fastest participant

In [ ]:
# ❌  Verbose: compute means manually, then loop to find the minimum

pid_rts = {}   # pid -> list of RTs
for t in trials:
    pid = t['participant_id']
    if pid not in pid_rts:
        pid_rts[pid] = []
    pid_rts[pid].append(float(t['rt']))

pid_means = {}
for pid, rt_list in pid_rts.items():
    pid_means[pid] = sum(rt_list) / len(rt_list)

fastest_pid  = None
fastest_mean = float('inf')
for pid, mean in pid_means.items():
    if mean < fastest_mean:
        fastest_mean = mean
        fastest_pid  = pid

print(f'Fastest: {fastest_pid}  mean RT = {fastest_mean:.3f} s')

In [ ]:
# ✅  Better: build the means dict cleanly, then use min() with key=
#
# min(iterable, key=function) returns the item for which function() is smallest.
# No manual tracking of 'current best' needed.

all_pids  = sorted(set(pids))
pid_means = {
    pid: np.mean([rt for rt, p in zip(rts, pids) if p == pid])
    for pid in all_pids
}

fastest_pid = min(pid_means, key=pid_means.get)

print(f'Fastest: {fastest_pid}  mean RT = {pid_means[fastest_pid]:.3f} s')

# Same pattern works for max(), sorted(), etc.
slowest_pid = max(pid_means, key=pid_means.get)
print(f'Slowest: {slowest_pid}  mean RT = {pid_means[slowest_pid]:.3f} s')

> **Why does it matter?**  
> The `key=` argument is one of the most useful things in Python.
> It works with `min()`, `max()`, `sorted()`, and more.
> Once you know it, you'll use it constantly.

---
## Example 5 — Normalising RTs (0–1 range)

In [ ]:
# ❌  Verbose: compute min/max first, then loop to apply the formula

rt_min = rts[0]
rt_max = rts[0]
for rt in rts:
    if rt < rt_min:
        rt_min = rt
    if rt > rt_max:
        rt_max = rt

rts_norm_loop = []
for rt in rts:
    rts_norm_loop.append((rt - rt_min) / (rt_max - rt_min))

print(f'min: {min(rts_norm_loop):.3f}  max: {max(rts_norm_loop):.3f}')
print(rts_norm_loop[:5])

In [ ]:
# ✅  Better: list comprehension with built-in min/max

rt_min = min(rts)
rt_max = max(rts)
rts_norm_comp = [(rt - rt_min) / (rt_max - rt_min) for rt in rts]

print(rts_norm_comp[:5])

In [ ]:
# ✅✅  Best: NumPy — the formula applies to the whole array at once
#
# rts_np is already a NumPy array, so arithmetic operators work element-wise.
# No loop, no comprehension — just the formula as you'd write it on paper.

rts_norm_np = (rts_np - rts_np.min()) / (rts_np.max() - rts_np.min())

print(rts_norm_np[:5])
print(f'min: {rts_norm_np.min():.3f}  max: {rts_norm_np.max():.3f}')

assert np.allclose(rts_norm_comp, rts_norm_np)
print('All approaches agree ✓')

> **Why does it matter?**  
> The NumPy version reads exactly like the mathematical formula: `(x − min) / (max − min)`.
> No translation into a loop required. This is what people mean by *expressive* code —
> the code looks like the thing it computes.

---
## Summary

| # | Task | Verbose approach | Elegant approach |
|---|------|-----------------|------------------|
| 1 | Unit conversion | `for` loop + `.append()` | `array * 1000` (NumPy) |
| 2 | Group means | One list per group, manual mean | dict comprehension |
| 3 | Count Trues | `for` + counter | `sum(booleans)` / `np.mean()` |
| 4 | Find min/max item | Loop tracking current best | `min(dict, key=dict.get)` |
| 5 | Normalise | Loop over formula | `(arr - arr.min()) / (arr.max() - arr.min())` |

**The general principle:** if you find yourself writing a `for` loop to do arithmetic or comparisons on a list of numbers, ask yourself — does NumPy have a function for this? It almost always does.